# Strategy Backtest v2 — point-in-time, next-open fills

Supersedes the v1 notebook `strategy_backtest.ipynb` (retired 2026-09-25; a copy is in the v1 backup folder `Stock Analysis Backup 2026-09-24`). Everything here is recomputed from raw Alpaca bars with the
**current** rules via `backtest_engine.py`:

- Features at date *t* use only bars ≤ *t* (Wilder RSI, trailing-window Force Index/OBV scaling, confirmed Fibonacci swings).
- Decision at the **close** of day *t*, fill at the **open** of *t+1*; 0.1% cost per side; partial intraday bars dropped.
- **Fundamentals and sentiment have no history → treated as unavailable** (technical + point-in-time relative strength only).
  Daily snapshots now accumulate in `Reports/factor_history.csv` so they can be tested later.
- Universe = `sector_mapping.tradable_symbols` (78 stocks; QQQ removed and used as a benchmark). A stock becomes eligible once it
  has 200 bars (late IPOs such as CRWV/CRCL/FIG enter only then).
- **Selection/survivorship bias:** the universe was hand-picked in 2026 with hindsight (many 2024–26 winners: MU, RKLB, HOOD, IREN,
  CIFR, ...). Absolute returns of every universe-based strategy — and of the equal-weight universe benchmark — are inflated by this.
  SPY/QQQ buy-and-hold are the only bias-free benchmarks.
- In-sample (parameter choice) = 2024-09-17 → 2025-09-16; out-of-sample = 2025-09-17 → latest complete bar. Each period is simulated
  separately starting flat with 1.0 of capital. Parameter grids are small and declared up front.

### Setup
Load cached daily bars and build the point-in-time scores used by every candidate below.

In [1]:
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

import backtest_engine as be

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 200, "display.max_columns", 50, "display.width", 250)

bars, dropped_partial = be.load_bars(refresh=True)
tech = be.build_technical(bars)
close_all, open_all = be.wide(bars, "Close"), be.wide(bars, "Open")
syms = [s for s in be.TRADABLE if s in close_all.columns]
C, O = close_all[syms], open_all[syms]

score_tech = be.wide(tech, "Technical_Score").reindex(index=C.index, columns=syms)
elig = be.bool_wide(tech, "eligible", C.index, syms)
atr = be.wide(tech, "atr").reindex(index=C.index, columns=syms)
rs_score, sector_rs = be.relative_strength(close_all, syms)
vol63 = C.pct_change().rolling(63).std()
regime = be.regime_series(close_all)
earnings = be.load_earnings()
block, reaction, is_earn = be.earnings_matrices(C.index, syms, earnings)
weekly = be.weekly_rebalance_days(C.index)
PERIODS = {"IS": (be.IS_START, be.IS_END), "OOS": (be.OOS_START, None), "Full": (be.IS_START, None)}
print(f"Bars {C.index.min():%Y-%m-%d} → {C.index.max():%Y-%m-%d}; partial last bar dropped: {dropped_partial}; "
      f"{len(syms)} tradable symbols; earnings events: {int(reaction.values.sum())}")

Bars 2023-06-01 → 2026-09-24; partial last bar dropped: False; 78 tradable symbols; earnings events: 614


### Helpers

In [2]:
results, rows = {}, []


def run(name, target, rebalance=None, params="", family=""):
    """Simulate `target` in each period and collect metrics."""
    tgt = target.reindex(index=close_all.index, columns=close_all.columns).fillna(0.0)
    results[name] = {}
    for per, (s, e) in PERIODS.items():
        res = be.simulate(open_all, close_all, tgt, s, e, rebalance=rebalance)
        results[name][per] = res
        rows.append({**be.metrics(res, name), "Period": per, "Family": family, "Params": params})
    return results[name]


def table(names=None, periods=("IS", "OOS")):
    t = pd.DataFrame(rows)
    if names is not None:
        t = t[t["Strategy"].isin(names)]
    t = t[t["Period"].isin(periods)]
    cols = ["Strategy", "Period", "CAGR %", "Total Return %", "Sharpe", "Sortino", "Calmar", "Max DD %", "Exposure %",
            "Time in Market %", "Return per Invested Day (bp)", "Turnover x/yr", "Trades", "Win Rate % (closed, net)"]
    return t[cols].round(2).set_index(["Strategy", "Period"])


def is_metric(name, col="Sharpe"):
    t = pd.DataFrame(rows)
    return float(t[(t["Strategy"] == name) & (t["Period"] == "IS")][col].iloc[0])

### E. Benchmarks — SPY / QQQ buy & hold, equal-weight universe (true buy & hold, and daily-rebalanced for reference)

In [3]:
for b in ["SPY", "QQQ"]:
    t = pd.DataFrame(0.0, index=close_all.index, columns=close_all.columns)
    t[b] = 1.0
    run(f"E: {b} buy & hold", t, family="Benchmark")

# Equal-weight universe buy & hold: bought at each period's first open among stocks trading then, never rebalanced
results["E: EW universe buy & hold"] = {}
for per, (s, e) in PERIODS.items():
    t, alive = be.buy_and_hold_target(close_all, syms, s)
    res = be.simulate(open_all, close_all, t, s, e)
    results["E: EW universe buy & hold"][per] = res
    rows.append({**be.metrics(res, "E: EW universe buy & hold"), "Period": per, "Family": "Benchmark",
                 "Params": f"{len(alive)} stocks alive at start (late IPOs excluded), no rebalancing"})
ew_daily = C.notna().astype(float).div(C.notna().sum(axis=1), axis=0)
run("E: EW universe daily-rebalanced (reference)", ew_daily, rebalance=pd.Series(True, index=C.index), family="Benchmark",
    params="rebalanced to equal weight every day (NOT buy & hold)")
table([r for r in results if r.startswith("E:")])

CAGR %  Total Return %  Sharpe  Sortino  Calmar  Max DD %  Exposure %  Time in Market %  Return per Invested Day (bp)  Turnover x/yr  Trades  Win Rate % (closed, net)
Strategy                                    Period                                                                                                                                                                        
E: SPY buy & hold                           IS       18.29           18.13    0.96     1.45    0.98    -18.75      100.00            100.00                          7.41           1.01       1                       NaN
                                            OOS      17.33           17.71    1.31     1.92    1.95     -8.88      100.00            100.00                          6.67           0.98       1                       NaN
E: QQQ buy & hold                           IS       24.92           24.70    1.06     1.60    1.09    -22.77      100.00            100.00                          9.92           1.01       1                       NaN
                                            OOS      25.42           25.99    1.25     1.85    2.12    -11.96      100.00            100.00                          9.76           0.98       1                       NaN
E: EW universe buy & hold                   IS      103.85          102.71    1.88     2.89    2.80    -37.14      100.00            100.00                         31.89           1.01      75                       NaN
                                            OOS      23.48           24.00    0.79     1.14    0.92    -25.52      100.00            100.00                         10.63           0.98      78                       NaN
E: EW universe daily-rebalanced (reference) IS      100.18           99.08    2.09     3.21    3.18    -31.46      100.00            100.00                         30.20           7.37      78                       NaN
                                            OOS      23.22           23.74    0.82     1.20    0.87    -26.74      100.00            100.00                         10.22           7.58      78                       NaN

### A. Current rules recomputed honestly (technical only, BUY > 20 / SELL < −25), equal 1/N slots

Each stock owns a 1/78 slot of equity at entry (cash when out), as in the old per-stock portfolio.

In [4]:
N_UNI = len(syms)
A_states = be.threshold_states(score_tech, elig, C, atr)
run("A: Current rules 20/-25 (1/N slots)", A_states / N_UNI, params="buy>20, sell<-25", family="A")

# Threshold sensitivity (pre-declared grid) — chosen on IS Sharpe only
sens = []
for buy in [10, 20, 30, 40]:
    for sell in [-15, -25, -35]:
        st = be.threshold_states(score_tech, elig, C, atr, buy=buy, sell=sell)
        tgt = (st / N_UNI).reindex(columns=close_all.columns).fillna(0.0)
        for per in ["IS", "OOS"]:
            m = be.metrics(be.simulate(open_all, close_all, tgt, *PERIODS[per]))
            sens.append({"BUY >": buy, "SELL <": sell, "Period": per, **{k: m[k] for k in ["CAGR %", "Sharpe", "Max DD %", "Exposure %", "Trades"]}})
sens = pd.DataFrame(sens)
sens_view = sens.pivot_table(index=["BUY >", "SELL <"], columns="Period", values=["Sharpe", "CAGR %", "Max DD %"]).round(2)
best_thr = sens[sens["Period"] == "IS"].sort_values("Sharpe", ascending=False).iloc[0]
BUY_T, SELL_T = int(best_thr["BUY >"]), int(best_thr["SELL <"])
print(f"IS-best thresholds: BUY > {BUY_T}, SELL < {SELL_T}")
A_t_states = be.threshold_states(score_tech, elig, C, atr, buy=BUY_T, sell=SELL_T)
run(f"A-tuned: thresholds {BUY_T}/{SELL_T} (1/N slots)", A_t_states / N_UNI, params=f"buy>{BUY_T}, sell<{SELL_T} (IS-chosen)", family="A")
AK10_name = "A-K10: 20/-25, max 10 positions @10%"
run(AK10_name, be.cap_positions(A_states, score_tech, 10) / 10, params="buy>20, sell<-25, top-10 by score", family="A")
sens_view

IS-best thresholds: BUY > 10, SELL < -25


CAGR %       Max DD %        Sharpe     
Period           IS   OOS       IS    OOS     IS  OOS
BUY > SELL <                                         
10    -35     62.07 13.32   -25.33 -21.27   1.88 0.61
      -25     61.52  7.60   -23.10 -19.04   1.95 0.43
      -15     59.00  7.11   -21.26 -18.91   1.92 0.42
20    -35     52.77 12.96   -25.13 -20.62   1.73 0.61
      -25     53.25  7.12   -22.57 -18.02   1.83 0.42
      -15     53.45  6.88   -21.04 -18.21   1.87 0.42
30    -35     47.10 14.48   -24.92 -20.35   1.62 0.67
      -25     48.76  8.36   -22.07 -17.65   1.75 0.49
      -15     48.95  7.66   -20.68 -17.90   1.78 0.46
40    -35     44.31 14.59   -23.84 -19.37   1.59 0.69
      -25     46.63  8.70   -20.55 -16.85   1.75 0.52
      -15     46.69  8.65   -19.21 -16.91   1.78 0.52

#### Per-stock view of rule A (whole period)

In [5]:
# Per-stock view of rule A over the full period (no trimming of top/bottom stocks)
per_stock = []
for s in syms:
    t = pd.DataFrame(0.0, index=close_all.index, columns=close_all.columns)
    t[s] = A_states[s]
    res = be.simulate(open_all, close_all, t, be.IS_START)
    first = C[s][C.index >= pd.Timestamp(be.IS_START)].first_valid_index()
    bh = C[s].iloc[-1] / O[s].loc[first] * (1 - be.COST) ** 2 - 1
    m = be.metrics(res, s)
    per_stock.append({"Symbol": s, "Strategy %": m["Total Return %"], "Buy & Hold % (from first open)": bh * 100,
                      "Max DD %": m["Max DD %"], "Exposure %": m["Exposure %"], "Closed trades": m["Closed Trades"],
                      "Win rate % (closed, net)": m["Win Rate % (closed, net)"], "First bar": C[s].first_valid_index().date()})
per_stock = pd.DataFrame(per_stock).set_index("Symbol").round(1)
per_stock["Beat B&H"] = per_stock["Strategy %"] > per_stock["Buy & Hold % (from first open)"]
per_stock.to_csv(be.REPORTS_DIR / "strategy_per_stock_v2.csv")
print(f"Rule A beat buy & hold in {per_stock['Beat B&H'].mean():.0%} of {len(per_stock)} stocks (full period, all stocks kept)")

Rule A beat buy & hold in 31% of 78 stocks (full period, all stocks kept)


### B. A + regime filter (new buys only when SPY > 200-day MA) + ATR trailing stop

Stop: exit when close < highest close since entry − k × ATR(14); after a stop the stock needs a fresh BUY (score back ≤ 20 then > 20).
k ∈ {2, 3} chosen on IS Sharpe. Also a concentrated variant: max 10 positions at 10% each (filled by highest score).

In [6]:
best_B = None
for k in [2, 3]:
    stB = be.threshold_states(score_tech, elig, C, atr, regime=regime, atr_k=k)
    name = f"B: 20/-25 + regime + {k}xATR stop (1/N slots)"
    run(name, stB / N_UNI, params=f"buy>20, sell<-25, SPY>200DMA for entries, {k}xATR trail", family="B")
    if best_B is None or is_metric(name) > is_metric(best_B[0]):
        best_B = (name, k, stB)
B_name, B_k, B_states = best_B
B10_name = f"B-K10: regime + {B_k}xATR stop, max 10 @10%"
run(B10_name, be.cap_positions(B_states, score_tech, 10) / 10, params=f"as B (k={B_k}), top-10 by score", family="B")
table([r for r in results if r.startswith(("A", "B"))])

CAGR %  Total Return %  Sharpe  Sortino  Calmar  Max DD %  Exposure %  Time in Market %  Return per Invested Day (bp)  Turnover x/yr  Trades  Win Rate % (closed, net)
Strategy                                    Period                                                                                                                                                                        
A: Current rules 20/-25 (1/N slots)         IS       53.25           52.74    1.83     2.72    2.36    -22.57       69.67            100.00                         26.12           7.20     313                     28.69
                                            OOS       7.12            7.26    0.42     0.58    0.40    -18.02       62.76            100.00                          5.86           8.60     367                     20.62
A-tuned: thresholds 10/-25 (1/N slots)      IS       61.52           60.91    1.95     2.92    2.66    -23.10       73.14            100.00                         27.93           8.48     363                     27.67
                                            OOS       7.60            7.76    0.43     0.60    0.40    -19.04       67.48            100.00                          5.85          10.68     451                     19.85
A-K10: 20/-25, max 10 positions @10%        IS       27.96           27.71    0.86     1.25    0.63    -44.03       98.50            100.00                         12.60          10.79      64                     31.48
                                            OOS      11.06           11.29    0.46     0.63    0.31    -35.83       98.19            100.00                          7.37           8.39      54                     31.82
B: 20/-25 + regime + 2xATR stop (1/N slots) IS       23.47           23.27    1.69     2.52    2.04    -11.49       41.16             90.00                         21.14          11.44     464                     39.05
                                            OOS       1.68            1.71    0.20     0.27    0.16    -10.38       36.48            100.00                          2.62          12.63     518                     33.81
B: 20/-25 + regime + 3xATR stop (1/N slots) IS       39.88           39.51    1.89     2.82    2.24    -17.81       54.33             94.40                         25.80           9.47     396                     34.71
                                            OOS       8.80            8.98    0.56     0.77    0.58    -15.28       50.24            100.00                          7.94          10.70     445                     26.65
B-K10: regime + 3xATR stop, max 10 @10%     IS       10.47           10.38    0.48     0.64    0.25    -42.69       88.30             94.40                          6.66          18.02     101                     30.77
                                            OOS      22.35           22.84    0.70     0.97    0.67    -33.30       99.10            100.00                         11.33          16.77     102                     30.43

### C. Cross-sectional ranking — weekly top-N by point-in-time score

score = w × Technical_Score + (1 − w) × RS_score (both −100..100; RS = cross-sectional percentile of stock-vs-sector-ETF and
sector-ETF-vs-SPY excess returns over 21/63/126 days, 60/40). Weekly: decide at the last close of the week, fill next open.
Names need score > 0; max 40% of names per sector (`sector_mapping.symbol_sector`); inverse-63-day-volatility weights; exposure =
names held / N. Regime variant: no new names while SPY < 200-day MA. Grid: N ∈ {5, 10} × regime {off, on} × w ∈ {0.3, 0.5, 0.7}.

In [7]:
c_grid = []
for n in [5, 10]:
    for use_reg in [False, True]:
        for w in [0.3, 0.5, 0.7]:
            score = w * score_tech + (1 - w) * rs_score
            tgt = be.rank_targets(score, elig, vol63, n=n, regime=regime if use_reg else None, rebalance_days=weekly)
            name = f"C: top{n} w_tech={w} regime={'on' if use_reg else 'off'}"
            run(name, tgt, rebalance=weekly, params=f"N={n}, w_tech={w}, regime={use_reg}, sector cap 40%, inv-vol", family="C")
            c_grid.append(name)
c_table = table(c_grid)
C_name = max(c_grid, key=is_metric)
print(f"IS-best ranking config: {C_name}")
c_table

IS-best ranking config: C: top10 w_tech=0.5 regime=off


CAGR %  Total Return %  Sharpe  Sortino  Calmar  Max DD %  Exposure %  Time in Market %  Return per Invested Day (bp)  Turnover x/yr  Trades  Win Rate % (closed, net)
Strategy                       Period                                                                                                                                                                        
C: top5 w_tech=0.3 regime=off  IS      128.16          126.67    2.00     3.10    3.44    -37.22      100.00            100.00                         37.12          50.08     112                     46.73
                               OOS       5.88            6.00    0.34     0.47    0.18    -32.30      100.00            100.00                          5.04          51.77     119                     39.47
C: top5 w_tech=0.5 regime=off  IS      135.94          134.34    2.21     3.45    3.74    -36.33      100.00            100.00                         37.78          52.73     124                     48.74
                               OOS      49.94           51.15    1.35     2.10    2.16    -23.12      100.00            100.00                         18.40          57.59     136                     48.85
C: top5 w_tech=0.7 regime=off  IS      118.32          116.97    2.03     3.18    2.94    -40.28       99.62            100.00                         34.82          59.88     144                     50.36
                               OOS      75.95           77.93    1.90     3.02    5.32    -14.28      100.00            100.00                         24.53          62.31     154                     53.02
C: top5 w_tech=0.3 regime=on   IS      114.35          113.05    1.91     2.97    3.35    -34.16       94.58            100.00                         36.31          45.25      98                     46.24
                               OOS      -2.83           -2.89    0.11     0.15   -0.09    -33.10       99.30            100.00                          1.58          49.74     115                     38.18
C: top5 w_tech=0.5 regime=on   IS      119.27          117.91    2.11     3.30    3.47    -34.37       90.65            100.00                         38.11          48.12     109                     48.08
                               OOS      39.00           39.91    1.14     1.77    1.69    -23.12       99.30            100.00                         15.45          54.67     129                     47.58
C: top5 w_tech=0.7 regime=on   IS      119.46          118.09    2.11     3.35    3.44    -34.72       90.25            100.00                         38.32          55.31     129                     51.61
                               OOS      63.17           64.76    1.68     2.65    3.14    -20.10       97.80            100.00                         21.99          59.13     146                     51.77
C: top10 w_tech=0.3 regime=off IS      143.39          141.68    2.42     3.77    4.29    -33.39      100.00            100.00                         38.54          42.13     180                     52.35
                               OOS      27.32           27.93    0.94     1.38    1.24    -21.95      100.00            100.00                         11.46          38.94     181                     39.18
C: top10 w_tech=0.5 regime=off IS      159.49          157.53    2.63     4.13    4.84    -32.96      100.00            100.00                         40.96          46.56     201                     52.36
                               OOS      34.48           35.28    1.18     1.74    1.46    -23.64      100.00            100.00                         13.38          42.54     207                     41.62
C: top10 w_tech=0.7 regime=off IS       93.47           92.46    2.11     3.04    2.60    -35.99       98.83            100.00                         28.87          50.67     231                     50.23
                               OOS      43.55           44.59    1.50     2.24    2.87    -15.19      100.00            100.0

### D. Earnings

**D1** flat-through-earnings overlay: be flat over every report's reaction gap (exit at the open of the session before the reaction
session, re-enter at the reaction session's open). Applied to the IS-best of A/A-tuned/A-K10/B/B-K10/C.
**D2** standalone post-earnings drift: if the reaction session return (close R vs close R−1) ≥ +5%, buy next open, hold H sessions
(H ∈ {10, 20}, IS-chosen), max 10 names at 10%. Earnings history (`Reports/earnings_date.csv`) starts 2024-09-25 and is incomplete
for some symbols before Dec 2024, so early-IS earnings coverage is partial.

In [8]:
core = ["A: Current rules 20/-25 (1/N slots)", [n for n in results if n.startswith("A-tuned")][0], AK10_name,
        B_name, B10_name, C_name]
best_core = max(core, key=is_metric)
print(f"IS-best core strategy for the overlay: {best_core}")


def rebuild_target(name):
    if name == C_name:
        n = int(name.split("top")[1].split()[0]); w = float(name.split("w_tech=")[1].split()[0]); r_on = name.endswith("on")
        return be.rank_targets(w * score_tech + (1 - w) * rs_score, elig, vol63, n=n, regime=regime if r_on else None,
                               rebalance_days=weekly), weekly
    if name.startswith("A: "):
        return A_states / N_UNI, None
    if name.startswith("A-tuned"):
        return A_t_states / N_UNI, None
    if name.startswith("A-K10"):
        return be.cap_positions(A_states, score_tech, 10) / 10, None
    if name == B_name:
        return B_states / N_UNI, None
    if name == B10_name:
        return be.cap_positions(B_states, score_tech, 10) / 10, None


base_t, base_reb = rebuild_target(best_core)
D1_name = f"D1: {best_core.split(':')[0]} + flat through earnings"
run(D1_name, base_t * (~block).astype(float), rebalance=base_reb, params=f"{best_core} with earnings overlay", family="D")
if best_core != C_name:  # also show the overlay on the ranking strategy for reference
    ct, creb = rebuild_target(C_name)
    run(f"D1: C + flat through earnings", ct * (~block).astype(float), rebalance=creb, params=f"{C_name} with earnings overlay", family="D")

d2 = []
for H in [10, 20]:
    name = f"D2: post-earnings drift >=5%, hold {H}d"
    run(name, be.pead_targets(reaction, C, O, threshold=0.05, hold=H, k=10), params=f"reaction>=+5%, hold {H}, max 10 @10%", family="D")
    d2.append(name)
D2_name = max(d2, key=is_metric)
table([n for n in results if n.startswith("D")])

IS-best core strategy for the overlay: C: top10 w_tech=0.5 regime=off


CAGR %  Total Return %  Sharpe  Sortino  Calmar  Max DD %  Exposure %  Time in Market %  Return per Invested Day (bp)  Turnover x/yr  Trades  Win Rate % (closed, net)
Strategy                               Period                                                                                                                                                                        
D1: C + flat through earnings          IS      144.14          142.41    2.57     3.96    4.43    -32.54       98.38            100.00                         38.91          53.77     239                     55.02
                                       OOS      23.93           24.46    0.90     1.31    0.97    -24.58       98.37            100.00                         10.25          49.76     241                     41.56
D2: post-earnings drift >=5%, hold 10d IS       21.98           21.78    1.07     1.59    0.97    -22.67       33.50             77.60                         26.06          16.87      85                     56.10
                                       OOS      -6.72           -6.85   -0.32    -0.45   -0.51    -13.31       28.83             75.10                         -7.60          14.83      76                     43.42
D2: post-earnings drift >=5%, hold 20d IS        6.73            6.67    0.38     0.51    0.21    -32.63       50.34             90.00                          7.79          12.63      67                     55.93
                                       OOS       2.24            2.28    0.21     0.30    0.14    -15.82       47.53             96.11                          3.84          12.46      66                     45.90

### Robustness for the ranking family: drawdown brake + walk-forward re-selection

- **DD brake** (pre-declared 15% / 0.5): while the strategy is >15% below its peak, new target weights are halved. Tested on the
  IS-best ranking config; adopted only if it improves IS Sharpe.
- **Walk-forward**: every 6 months re-pick the C config (same 12-config grid) on the trailing 12 months, trade it for the next 6
  months. Blocks: 2025-09-17→2026-03-16 (picked on 2024-09-17→2025-09-16) and 2026-03-17→end (picked on 2025-03-17→2026-03-16).

In [9]:
c_t, c_reb = None, weekly
n_c = int(C_name.split("top")[1].split()[0]); w_c = float(C_name.split("w_tech=")[1].split()[0]); reg_c = C_name.endswith("on")
c_t = be.rank_targets(w_c * score_tech + (1 - w_c) * rs_score, elig, vol63, n=n_c, regime=regime if reg_c else None, rebalance_days=weekly)
DD_name = f"C + DD brake 15%/0.5 ({C_name.split(': ')[1]})"
results[DD_name] = {}
for per, (s, e) in PERIODS.items():
    res = be.simulate(open_all, close_all, c_t.reindex(columns=close_all.columns).fillna(0.0), s, e, rebalance=weekly, dd_brake=(0.15, 0.5))
    results[DD_name][per] = res
    rows.append({**be.metrics(res, DD_name), "Period": per, "Family": "C", "Params": f"{C_name} + halve weights while DD > 15%"})
dd_adopted = is_metric(DD_name) > is_metric(C_name)
print(f"DD brake improves IS Sharpe: {dd_adopted}")

grid_targets = {}
for n in [5, 10]:
    for use_reg in [False, True]:
        for w in [0.3, 0.5, 0.7]:
            grid_targets[(n, use_reg, w)] = be.rank_targets(w * score_tech + (1 - w) * rs_score, elig, vol63, n=n,
                                                            regime=regime if use_reg else None, rebalance_days=weekly
                                                            ).reindex(columns=close_all.columns).fillna(0.0)
blocks = [(("2024-09-17", "2025-09-16"), ("2025-09-17", "2026-03-16")), (("2025-03-17", "2026-03-16"), ("2026-03-17", None))]
wf_target = pd.DataFrame(0.0, index=close_all.index, columns=close_all.columns)
wf_picks = []
for (tr_s, tr_e), (te_s, te_e) in blocks:
    best = max(grid_targets, key=lambda k: be.metrics(be.simulate(open_all, close_all, grid_targets[k], tr_s, tr_e, rebalance=weekly))["Sharpe"])
    wf_picks.append({"train": f"{tr_s}→{tr_e}", "test": f"{te_s}→{te_e or 'end'}", "N": best[0], "regime": best[1], "w_tech": best[2]})
    lo = close_all.index.searchsorted(pd.Timestamp(te_s)) - 1          # decision on the close before the block starts
    hi = len(close_all.index) if te_e is None else close_all.index.searchsorted(pd.Timestamp(te_e), side="right")
    wf_target.iloc[lo:hi] = grid_targets[best].iloc[lo:hi].values
WF_name = "C walk-forward (6m re-selection)"
res = be.simulate(open_all, close_all, wf_target, be.OOS_START, None, rebalance=weekly | pd.Series(close_all.index.isin(
    [close_all.index[close_all.index.searchsorted(pd.Timestamp("2026-03-17")) - 1]]), index=close_all.index))
results[WF_name] = {"OOS": res}
rows.append({**be.metrics(res, WF_name), "Period": "OOS", "Family": "C", "Params": "; ".join(f"{p['test']}: N={p['N']}, regime={p['regime']}, w={p['w_tech']}" for p in wf_picks)})
print(pd.DataFrame(wf_picks).to_string(index=False))
table([C_name, DD_name, WF_name])

DD brake improves IS Sharpe: True


                train                  test  N  regime  w_tech
2024-09-17→2025-09-16 2025-09-17→2026-03-16 10   False    0.50
2025-03-17→2026-03-16        2026-03-17→end  5    True    0.70


CAGR %  Total Return %  Sharpe  Sortino  Calmar  Max DD %  Exposure %  Time in Market %  Return per Invested Day (bp)  Turnover x/yr  Trades  Win Rate % (closed, net)
Strategy                                           Period                                                                                                                                                                        
C: top10 w_tech=0.5 regime=off                     IS      159.49          157.53    2.63     4.13    4.84    -32.96      100.00            100.00                         40.96          46.56     201                     52.36
                                                   OOS      34.48           35.28    1.18     1.74    1.46    -23.64      100.00            100.00                         13.38          42.54     207                     41.62
C + DD brake 15%/0.5 (top10 w_tech=0.5 regime=off) IS      157.40          155.47    2.81     4.47    6.28    -25.08       87.45            100.00                         45.92          42.61     201                     52.36
                                                   OOS      30.60           31.29    1.11     1.63    1.42    -21.56       90.69            100.00                         13.32          39.09     207                     41.12
C walk-forward (6m re-selection)                   OOS      47.50           48.64    1.38     2.08    2.58    -18.42       97.80            100.00                         17.84          52.14     177                     47.67

### Comparison & winner (chosen by OUT-OF-SAMPLE Sharpe, Calmar/max DD as tie-breakers, vs SPY/QQQ)

In [10]:
candidates = list(dict.fromkeys(core + [D1_name] + [n for n in results if n.startswith("D1: C")] + [D2_name, DD_name]))
benchmarks = [n for n in results if n.startswith("E:")]
full = pd.DataFrame(rows)
full.round(3).to_csv(be.REPORTS_DIR / "strategy_comparison_v2.csv", index=False)
sens.round(3).to_csv(be.REPORTS_DIR / "strategy_threshold_sensitivity_v2.csv", index=False)

oos = full[full["Period"] == "OOS"].set_index("Strategy")
ranked = oos.loc[candidates].sort_values(["Sharpe", "Calmar"], ascending=False)
winner = ranked.index[0]
qqq, spy = oos.loc["E: QQQ buy & hold"], oos.loc["E: SPY buy & hold"]
beats_qqq = (oos.loc[winner, "Sharpe"] > qqq["Sharpe"]) and (oos.loc[winner, "Calmar"] > qqq["Calmar"])
print(f"WINNER (OOS Sharpe): {winner}")
print(f"  OOS Sharpe {oos.loc[winner, 'Sharpe']:.2f} vs QQQ {qqq['Sharpe']:.2f} / SPY {spy['Sharpe']:.2f}; "
      f"Calmar {oos.loc[winner, 'Calmar']:.2f} vs QQQ {qqq['Calmar']:.2f}; MaxDD {oos.loc[winner, 'Max DD %']:.1f}% vs QQQ {qqq['Max DD %']:.1f}%")
print("  Beats QQQ buy & hold on a risk-adjusted basis OOS:" , "YES" if beats_qqq else "NO")
table(candidates + benchmarks + [WF_name])

WINNER (OOS Sharpe): C: top10 w_tech=0.5 regime=off
  OOS Sharpe 1.18 vs QQQ 1.25 / SPY 1.31; Calmar 1.46 vs QQQ 2.12; MaxDD -23.6% vs QQQ -12.0%
  Beats QQQ buy & hold on a risk-adjusted basis OOS: NO


CAGR %  Total Return %  Sharpe  Sortino  Calmar  Max DD %  Exposure %  Time in Market %  Return per Invested Day (bp)  Turnover x/yr  Trades  Win Rate % (closed, net)
Strategy                                           Period                                                                                                                                                                        
E: SPY buy & hold                                  IS       18.29           18.13    0.96     1.45    0.98    -18.75      100.00            100.00                          7.41           1.01       1                       NaN
                                                   OOS      17.33           17.71    1.31     1.92    1.95     -8.88      100.00            100.00                          6.67           0.98       1                       NaN
E: QQQ buy & hold                                  IS       24.92           24.70    1.06     1.60    1.09    -22.77      100.00            100.00                          9.92           1.01       1                       NaN
                                                   OOS      25.42           25.99    1.25     1.85    2.12    -11.96      100.00            100.00                          9.76           0.98       1                       NaN
E: EW universe buy & hold                          IS      103.85          102.71    1.88     2.89    2.80    -37.14      100.00            100.00                         31.89           1.01      75                       NaN
                                                   OOS      23.48           24.00    0.79     1.14    0.92    -25.52      100.00            100.00                         10.63           0.98      78                       NaN
E: EW universe daily-rebalanced (reference)        IS      100.18           99.08    2.09     3.21    3.18    -31.46      100.00            100.00                         30.20           7.37      78                       NaN
                                                   OOS      23.22           23.74    0.82     1.20    0.87    -26.74      100.00            100.00                         10.22           7.58      78                       NaN
A: Current rules 20/-25 (1/N slots)                IS       53.25           52.74    1.83     2.72    2.36    -22.57       69.67            100.00                         26.12           7.20     313                     28.69
                                                   OOS       7.12            7.26    0.42     0.58    0.40    -18.02       62.76            100.00                          5.86           8.60     367                     20.62
A-tuned: thresholds 10/-25 (1/N slots)             IS       61.52           60.91    1.95     2.92    2.66    -23.10       73.14            100.00                         27.93           8.48     363                     27.67
                                                   OOS       7.60            7.76    0.43     0.60    0.40    -19.04       67.48            100.00                          5.85          10.68     451                     19.85
A-K10: 20/-25, max 10 positions @10%               IS       27.96           27.71    0.86     1.25    0.63    -44.03       98.50            100.00                         12.60          10.79      64                     31.48
                                                   OOS      11.06           11.29    0.46     0.63    0.31    -35.83       98.19            100.00                          7.37           8.39      54                     31.82
B: 20/-25 + regime + 3xATR stop (1/N slots)        IS       39.88           39.51    1.89     2.82    2.24    -17.81       54.33             94.40                         25.80           9.47     396                     34.71
                                                   OOS       8.80            8.98    0.56     0.77    0.58    -15.28       50.24            100.00                          7.94          10.70     445                  

### Equity curves

In [11]:
# Equity curves: full period (IS params applied throughout; dashed line = OOS start) and OOS-only (fresh start)
show = list(dict.fromkeys(["A: Current rules 20/-25 (1/N slots)", B_name, C_name, winner, "E: SPY buy & hold", "E: QQQ buy & hold",
                           "E: EW universe buy & hold"]))
fig, axes = plt.subplots(1, 2, figsize=(16, 6.5))
for ax, per in zip(axes, ["Full", "OOS"]):
    for n in show:
        eq = results[n][per]["equity"]
        style = dict(lw=2.6) if n == winner else dict(lw=1.3, ls="--" if n.startswith("E:") else "-")
        ax.plot(eq.index, eq.values, label=f"{n} ({(eq.iloc[-1] - 1) * 100:+.0f}%)", **style)
    if per == "Full":
        ax.axvline(pd.Timestamp(be.OOS_START), color="grey", ls=":", lw=1)
        ax.text(pd.Timestamp(be.OOS_START), ax.get_ylim()[1] * 0.97, "  out-of-sample →", color="grey")
    ax.set_title(f"{'Full period (IS-chosen parameters)' if per == 'Full' else 'Out-of-sample only, fresh start'}")
    ax.set_ylabel("Equity (start = 1.0)")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=7.5, loc="upper left")
fig.suptitle("Strategy backtest v2 — point-in-time signals, next-open fills, 0.1%/side. Universe is hand-picked (selection bias).", fontsize=11)
fig.tight_layout()
CHART = be.REPORTS_DIR / "strategy_equity_curves_v2.png"
fig.savefig(CHART, dpi=130)
print(f"Saved {CHART.name}, strategy_comparison_v2.csv, strategy_threshold_sensitivity_v2.csv, strategy_per_stock_v2.csv")

Saved strategy_equity_curves_v2.png, strategy_comparison_v2.csv, strategy_threshold_sensitivity_v2.csv, strategy_per_stock_v2.csv
